## <span style='color:green'>2.0 Depature Data Frame </span>
#### Goals of the Project 
This is a Data cleaning project aimed at preparing Data for Machine learning 

In [1]:
import numpy  as np
import pandas as pd 
from matplotlib import pyplot as plt
import seaborn as sns 
import dateutil.parser as parser 
from datetime import datetime
import dateutil.parser as parser 
from datetime import datetime
from sklearn.linear_model import LinearRegression #to build a linear regression
%config InlineBackend.figure_format = 'retina'

### Stored DF retrieved Arrival_DF 
- The use of the %store -r Arrival_DF is to retrive the Data Frame from the DELI MACHINE LEARNING PROJECT PART1

In [2]:
%store -r Depature_DF
Depature_DF.shape

(902050, 16)

In [4]:
Depature_DF.head(1) 

,Date,Day of week,Weekend (0/1),Holiday (0/1),Festival,Overlap with Weekend,Extended weekend,boarded pax,Original Airport,Destination airport,Depature Datetime,Seat Capacity,traffic_type,Flight Number,Terminal,Airline Code
0,2018-12-31,Monday,0,0,0,0,0,166,DEL,BLR,2018-12-31 05:50:00,186.0,D,G8113,T2,G8


In [5]:
Depature_DF.columns # Looking at what columns we do have 

Index(['Date', 'Day of week', 'Weekend (0/1)', 'Holiday (0/1)', 'Festival',
       'Overlap with Weekend', 'Extended weekend', 'boarded pax',
       'Original Airport', 'Destination airport', 'Depature Datetime',
       'Seat Capacity', 'traffic_type', 'Flight Number', 'Terminal',
       'Airline Code'],
      dtype='object')

In [14]:
Depature_DF.dtypes # Checking out the available Data types 

Date                    datetime64[ns]
Day of week                     object
Weekend (0/1)                    int32
Holiday (0/1)                    int64
Festival                         int64
Overlap with Weekend             int32
Extended weekend                 int32
boarded pax                    float64
Original Airport                object
Destination airport             object
Depature Datetime               object
Seat Capacity                   object
traffic_type                    object
Flight Number                   object
Terminal                        object
Airline Code                    object
dtype: object

### 2. Converting Seat Capacity and boarded Pax into int for Pax ratio
- This section involves converting both the Seat Capacity and boarded Pax into an integer , in this way , a pax ratio will be calulated .
- We apply pd.to_numeric and downcasting it is converting a data type to a lower precision or a smaller range number.

In [15]:
Depature_DF["Seat Capacity"]=Depature_DF["Seat Capacity"].apply(pd.to_numeric,errors='coerce',downcast='integer')

In [16]:
Depature_DF["boarded pax"]=Depature_DF["boarded pax"].apply(pd.to_numeric,errors='coerce',downcast='integer')

### 3. Removing duplicated rows 
- This chapter aims at observing available duplicates and removing them 

In [17]:
duplicateRows=Depature_DF[Depature_DF.duplicated()]
duplicateRows.shape # Flight number and Flight date  time .. We do not have an duplicate values . 

(0, 16)

In [18]:
print("percentage of duplicates is :", (len(duplicateRows)/len(Depature_DF)*100))
#only 0.056 percent and will be deleted .. too small to make any different 

percentage of duplicates is : 0.0


In [19]:
Depature_DF=Depature_DF.drop_duplicates()
len(Depature_DF) # meangless since we do not have an duplicates in the first case. 

820486

### 4. Droping values where boarded passangers are zero

In [20]:
Depature_DF_0 = Depature_DF[Depature_DF["boarded pax"]==0.0]
pecentage0= len(Depature_DF_0)/len(Depature_DF)
pecentage0*100
#Arrival_DF = Arrival_DF.dropna(subset=["boarded pax"])
#Arrival_DF = Arrival_DF[Arrival_DF["boarded pax"]!=0]

0.2801997840304405

In [21]:
Depature_DF = Depature_DF.dropna(subset=["boarded pax"])
Depature_DF.shape # duplicates dropped hence we are left with fewer rows .

(820486, 16)

### 5.  Pax Ratio Calculation 

I am going to find the ratio of the Pax number over the Aircraft capacity ie : 
- Pax ratio is defined as the ratio of the boarded pax to the Aircraft Capacity
$$
PA_r=Paxt/Seats
$$
where PA_r stands for Passanger Aircraft Ratio 

In [22]:
Depature_DF.loc[:,"PAr"]=round((Depature_DF['boarded pax']/Depature_DF['Seat Capacity']),6)# round it to 4.sf
Depature_DF.shape

(820486, 17)

In [23]:
greate1=Depature_DF[(Depature_DF["PAr"] > 1) | (Depature_DF["PAr"] == float("inf"))] ## starting from here . 
greate1.shape

(45834, 17)

### Percentage of values greater than zero

In [24]:
greate1percent=(len(greate1)/len(Depature_DF))*100
print("%percent of those greater than 1 is : ", greate1percent)

%percent of those greater than 1 is :  5.586201348956594


### Saving DB containing values where the pax load is greater than 1 .

In [25]:
greate1.loc[:,"Flight Number"]=greate1.loc[:,"Flight Number"].str.replace(r'\s','')
greate1.loc[:,"Flight Number"]=greate1.loc[:,"Flight Number"].apply(str)
greate1.loc[:,"Aircraft Type"]=greate1.loc[:,"Flight Number"].str[-3:] # choosing only the first right three digits . 
#value_counts()
greate1.to_excel(r"C:\Users\enock.mugabi\AOBD_Historical_Data\Investigations\PAR_greaterthan1pending.xlsx")

C:\Users\enock.mugabi\AppData\Local\Temp\ipykernel_27900\1859778492.py:3: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  greate1.loc[:,"Aircraft Type"]=greate1.loc[:,"Flight Number"].str[-3:] # choosing only the first right three digits .


In [26]:
Depature_DF.drop(Depature_DF[(Depature_DF["PAr"] > 1) | (Depature_DF["PAr"] == float("inf"))].index, inplace=True)
Depature_DF.shape

(774652, 17)

In [27]:
Depature_DF["boarded pax"].sum()

121850458.0

#### Extracting Month and Week Number from the the Date format 

In [28]:
Depature_DF.loc[:,"Month"]=Depature_DF["Date"].dt.strftime('%B')
Depature_DF.loc[:,"WN"]=Depature_DF["Date"].dt.isocalendar().week
Depature_DF.loc[:,"Year"]=Depature_DF.loc[:,"Date"].dt.year

In [29]:
Depature_DF["Year"].unique() # Here we will use that for 2018 , 2019 and 2022 . to forsee that for 2023 . 

array([2018, 2019, 2022, 2023])

#### Chossing the most important rows For the to be trained DB

In [30]:
Depature_DF.columns

Index(['Date', 'Day of week', 'Weekend (0/1)', 'Holiday (0/1)', 'Festival',
       'Overlap with Weekend', 'Extended weekend', 'boarded pax',
       'Original Airport', 'Destination airport', 'Depature Datetime',
       'Seat Capacity', 'traffic_type', 'Flight Number', 'Terminal',
       'Airline Code', 'PAr', 'Month', 'WN', 'Year'],
      dtype='object')

In [35]:
Spaces=Depature_DF["traffic_type"]==' '
Depature_DF = Depature_DF[~Spaces]

In [36]:
Depature_DF["traffic_type"].unique() # We have to cut the data indo traffic types .. 

array(['D', 'I', 'J', 'DOM', 'INT'], dtype=object)

In [37]:
Depature_DF["traffic_type"]=Depature_DF["traffic_type"].replace('INT','I')
Depature_DF["traffic_type"]=Depature_DF["traffic_type"].replace('DOM','D')

#### Dropping Airport Destinations that where considred as Null in the DB

In [38]:
Depature_DF.dropna(subset=["Destination airport"]).copy() # Arrival_DF.dropna(subset=["Original Airport"]).copy

,Date,Day of week,Weekend (0/1),Holiday (0/1),Festival,Overlap with Weekend,Extended weekend,boarded pax,Original Airport,Destination airport,Depature Datetime,Seat Capacity,traffic_type,Flight Number,Terminal,Airline Code,PAr,Month,WN,Year
0,2018-12-31,Monday,0,0,0,0,0,166.0,DEL,BLR,2018-12-31 05:50:00,186.0,D,G8113,T2,G8,0.892473,December,1,2018
1,2018-12-30,Sunday,1,0,0,1,1,88.0,DEL,PAT,2018-12-30 15:10:00,186.0,D,G8149,T2,G8,0.473118,December,52,2018
2,2018-12-30,Sunday,1,0,0,1,1,157.0,DEL,AMD,2018-12-30 06:00:00,186.0,D,G8719,T2,G8,0.844086,December,52,2018
3,2018-12-30,Sunday,1,0,0,1,1,103.0,DEL,HYD,2018-12-30 07:30:00,186.0,D,G8423,T2,G8,0.553763,December,52,2018
4,2018-12-30,Sunday,1,0,0,1,1,167.0,DEL,BOM,2018-12-30 19:40:00,186.0,D,G8446,T2,G8,0.897849,December,52,2018
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
902045,2023-12-31,Sunday,1,0,0,1,1,173.0,DEL,AMD,2023-12-31 09:40:00,186.0,D,6E2264,T2,6E,0.930108,December,52,2023
902046,2023-12-31,Sunday,1,0,0,1,1,180.0,DEL,DMU,2023-12-31 10:15:00,186.0,D,6E2625,T2,6E,0.967742,December,52,2023
902047,2023-12-31,Sunday,1,0,0,1,1,180.0,DEL,SXR,2023-12-31 06:45:00,186.0,D,6E2981,T2,6E,0.967742,December,52,2023
902048,2023-12-31,Sunday,1,0,0,1,1,155.0,DEL,AJL,2023-12-31 12:00:00,186.0,D,6E5363,T3,6E,0.833333,December,52,2023


#### Dropping classes where have PAr as Nan

In [39]:
DD=Depature_DF.dropna(subset=["PAr"]).copy() # Arrival_DF.dropna(subset=["Original Airport"]).copy

In [40]:
DD.replace([np.inf, -np.inf], np.nan, inplace=True)

In [41]:
DD=Depature_DF.fillna(Depature_DF["PAr"])

In [44]:
Feature_Selection_Depature=DD 
    # This is being stored and then retrieved for feature selection 

In [45]:
%store  Feature_Selection_Depature

Stored 'Feature_Selection_Depature' (DataFrame)
